# PNCP — Silver & Gold com Delta Lake (Databricks Free Edition)

Compatível com **Serverless Compute** do Databricks Free Edition:
- Sem chamadas de rede externas (saída de internet é restrita no Free Edition): lê o Parquet Silver enviado para um **Unity Catalog Volume**.
- Sem `.cache()`, `.persist()` ou variáveis de ambiente (proibidos no Serverless); usa `dbutils.widgets`.

In [ ]:
try:
    dbutils.widgets.text("silver_volume_path", "/Volumes/workspace/default/pncp/silver/contratacoes")
    dbutils.widgets.text("catalog_schema", "workspace.default")
    silver_path = dbutils.widgets.get("silver_volume_path")
    target_schema = dbutils.widgets.get("catalog_schema")
except NameError:
    silver_path = "/Volumes/workspace/default/pncp/silver/contratacoes"
    target_schema = "workspace.default"

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

silver_batch = spark.read.parquet(silver_path)
silver_table = f"{target_schema}.silver_pncp_contratacoes"

if not spark.catalog.tableExists(silver_table):
    (
        silver_batch.write.format("delta")
        .mode("overwrite")
        .partitionBy("pub_year", "pub_month", "uf_sigla")
        .saveAsTable(silver_table)
    )
else:
    DeltaTable.forName(spark, silver_table).alias("target").merge(
        silver_batch.alias("src"),
        "target.numero_controle_pncp = src.numero_controle_pncp",
    ).whenMatchedUpdateAll(
        condition="target.data_atualizacao IS NULL OR src.data_atualizacao >= target.data_atualizacao"
    ).whenNotMatchedInsertAll().execute()

In [ ]:
gold_table = f"{target_schema}.gold_pncp_resumo_uf_mes"
silver_current = spark.table(silver_table)

economia = F.when(
    F.col("valor_total_homologado").isNotNull() & F.col("valor_total_estimado").isNotNull(),
    F.col("valor_total_estimado") - F.col("valor_total_homologado"),
).otherwise(F.lit(0.0))

gold_df = (
    silver_current.groupBy(
        "pub_year", "pub_month", "uf_sigla", "modalidade_id", "modalidade_nome"
    )
    .agg(
        F.count("numero_controle_pncp").alias("qtd_contratacoes"),
        F.count("valor_total_homologado").alias("qtd_homologadas"),
        F.round(F.sum(F.coalesce(F.col("valor_total_estimado"), F.lit(0.0))), 2).alias("total_estimado"),
        F.round(F.sum(F.coalesce(F.col("valor_total_homologado"), F.lit(0.0))), 2).alias("total_homologado"),
        F.round(F.sum(economia), 2).alias("economia_homologada"),
    )
)

gold_df.write.format("delta").mode("overwrite").saveAsTable(gold_table)

In [ ]:
spark.sql(f"DESCRIBE HISTORY {silver_table}").select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)
spark.sql(f"SELECT uf_sigla, modalidade_nome, qtd_contratacoes, total_estimado, total_homologado FROM {gold_table} ORDER BY total_estimado DESC LIMIT 15").show(truncate=False)